In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from xgboost import XGBRegressor
import optuna, mlflow
import mlflow.xgboost

In [2]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [3]:
df_train = pd.read_csv('../data/processed/feature_engineered_train.csv')
df_eval = pd.read_csv('../data/processed/feature_engineered_eval.csv')

target = 'price'
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_eval = df_eval.drop(columns=[target])
y_eval = df_eval[target]

In [5]:

mlflow.set_experiment('XGBoost Single Experiment')
mlflow.set_tracking_uri("../mlruns")

with mlflow.start_run(nested=False):

    params = {'n_estimators':500,
            'learning_rate':0.05,
            'max_depth':6,
            'subsample':0.8,
            'colsample_bytree':0.8,
            'random_state':42,
            'n_jobs':-1}

    xgb_model = xgb.XGBRegressor(**params)

    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_eval)

    mae = mean_absolute_error(y_eval.values, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval.values, y_pred))
    r2 = r2_score(y_eval.values, y_pred)

    print('XGBoost:\n')
    print(f'MAE: {mae:,.2f}')
    print(f'RMSE: {rmse:,.2f}')
    print(f'R2: {r2:,.4f}')


    mlflow.log_params(params)
    mlflow.log_metrics({'mae':mae, 'rmse':rmse, 'r2':r2})
    mlflow.set_tag("Training Info", "Basic LR model for iris data")




XGBoost:

MAE: 32,885.11
RMSE: 72,419.49
R2: 0.9595


In [13]:
# Optimization of the hyperparameters using Optuna and tracking of the model performances using MLflow

mlflow.set_experiment("XGBoost Hyperparameter Tuning Experiment")

def objective(trial): # optuna's objective function to minimize
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run : 
        model_params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "tree_method": "hist",
        }

        # train and evaluate model
        xgb_model = xgb.XGBRegressor(**model_params)
        xgb_model.fit(X_train, y_train)
        y_pred = xgb_model.predict(X_eval)
        mae = mean_absolute_error(y_eval.values, y_pred)
        rmse = np.sqrt(mean_squared_error(y_eval.values, y_pred))
        r2 = r2_score(y_eval.values, y_pred)
        print('XGBoost:\n')
        print(f'MAE: {mae:,.2f}')
        print(f'RMSE: {rmse:,.2f}')
        print(f'R2: {r2:,.4f}')

        mlflow.log_params(model_params) # save model parameters
        mlflow.log_metrics({'mae':mae, 'rmse':rmse, 'r2':r2}) # save metrics
        # log model (how?)
        trial.set_user_attr("run_id", child_run.info.run_id)

    return rmse


with mlflow.start_run(run_name='study') as run:

    n_trials = 5
    mlflow.log_param('n_trials', n_trials)

    # optimize hyperparameters
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    # save best model
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

    best_params = study.best_trial.params
    print('Best parameters : ', study.best_trial.params)
    


[I 2026-03-17 10:49:13,498] A new study created in memory with name: no-name-2d1e326b-6be2-4267-8f2f-8243a28cb5d0
[I 2026-03-17 10:49:26,333] Trial 0 finished with value: 74751.71343078664 and parameters: {'n_estimators': 413, 'max_depth': 6, 'learning_rate': 0.07503690166508065, 'subsample': 0.6376928843986447, 'colsample_bytree': 0.9702836980535854, 'min_child_weight': 7, 'gamma': 2.0258498323365783, 'reg_alpha': 2.2830496027808973, 'reg_lambda': 0.0033994690148261165}. Best is trial 0 with value: 74751.71343078664.


XGBoost:

MAE: 32,677.54
RMSE: 74,751.71
R2: 0.9568


[I 2026-03-17 10:49:31,674] Trial 1 finished with value: 76249.95247179888 and parameters: {'n_estimators': 338, 'max_depth': 3, 'learning_rate': 0.15763508849791358, 'subsample': 0.9851469409102268, 'colsample_bytree': 0.7647610443156299, 'min_child_weight': 5, 'gamma': 4.941846898422203, 'reg_alpha': 4.178241887018418e-08, 'reg_lambda': 7.863067431607156e-08}. Best is trial 0 with value: 74751.71343078664.


XGBoost:

MAE: 35,954.38
RMSE: 76,249.95
R2: 0.9551


[I 2026-03-17 10:49:46,282] Trial 2 finished with value: 74880.38971874239 and parameters: {'n_estimators': 427, 'max_depth': 7, 'learning_rate': 0.012005541878493394, 'subsample': 0.9033602477924987, 'colsample_bytree': 0.7891172913115182, 'min_child_weight': 3, 'gamma': 0.9173164540642093, 'reg_alpha': 0.0003988476686713019, 'reg_lambda': 0.27585750952985244}. Best is trial 0 with value: 74751.71343078664.


XGBoost:

MAE: 34,253.43
RMSE: 74,880.39
R2: 0.9567


[I 2026-03-17 10:49:56,797] Trial 3 finished with value: 77502.91108573816 and parameters: {'n_estimators': 312, 'max_depth': 7, 'learning_rate': 0.019095058620329824, 'subsample': 0.8626265134958638, 'colsample_bytree': 0.9673365624715906, 'min_child_weight': 9, 'gamma': 2.279485690029076, 'reg_alpha': 0.0004570977033792074, 'reg_lambda': 0.0001433354253798928}. Best is trial 0 with value: 74751.71343078664.


XGBoost:

MAE: 34,108.79
RMSE: 77,502.91
R2: 0.9536


[I 2026-03-17 10:50:08,985] Trial 4 finished with value: 73603.4931009403 and parameters: {'n_estimators': 361, 'max_depth': 8, 'learning_rate': 0.05075769699040559, 'subsample': 0.9468715161081269, 'colsample_bytree': 0.6746550902017974, 'min_child_weight': 9, 'gamma': 0.6964285012812993, 'reg_alpha': 6.324231752100331, 'reg_lambda': 0.42043044803520496}. Best is trial 4 with value: 73603.4931009403.


XGBoost:

MAE: 31,896.28
RMSE: 73,603.49
R2: 0.9581
Best parameters :  {'n_estimators': 361, 'max_depth': 8, 'learning_rate': 0.05075769699040559, 'subsample': 0.9468715161081269, 'colsample_bytree': 0.6746550902017974, 'min_child_weight': 9, 'gamma': 0.6964285012812993, 'reg_alpha': 6.324231752100331, 'reg_lambda': 0.42043044803520496}


In [17]:
# Evaluation of the best model founds


# train and evaluate model
xgb_model = xgb.XGBRegressor(**best_params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_eval)

mae = mean_absolute_error(y_eval.values, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval.values, y_pred))
r2 = r2_score(y_eval.values, y_pred)

print('XGBoost:\n')
print(f'MAE: {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')
print(f'R2: {r2:,.4f}')

# Log best model
with mlflow.start_run(run_name='best_xgboost_model'):
    mlflow.log_params(best_params)
    mlflow.log_metrics({'mae':mae, 'rmse':rmse, 'r2':r2}) 
    mlflow.xgboost.log_model(xgb_model, name='model')



XGBoost:

MAE: 31,618.19
RMSE: 70,953.79
R2: 0.9611


2026/03/17 10:54:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
